# 09 — Gelombang 1: sapuan, ablasi, pesaing, uji statistik (CPU)

Semua yang `reports/topvenue_gap_audit.md` sebut bisa dikerjakan tanpa GPU, dalam
satu run. Tidak bergantung pada notebook 10/11, jadi ketiganya bisa jalan paralel.

| Fase | Menjawab |
|---|---|
| **A. Kedalaman kalibrasi** 10→200 | kapan metodenya bekerja — **kurva**, bukan anekdot |
| **B. Fraksi held-out** 10/30/50% | berapa banyak kelas boleh tanpa data |
| **C. Ablasi** 5 sumbu | bagian mana yang sebenarnya bekerja |
| **D. Pesaing** Fuzzy Classwise, PAS/Interp-Q | pesaing konseptual terdekat (§7: paling berbahaya) |
| **E. Uji statistik** | berpasangan + Holm + effect size |
| **F. Runtime** | klaim praktis: nol training |

**Hasil disimpan ke Drive setelah SETIAP fase**, bukan hanya di akhir — runtime yang
mati di fase D tidak boleh menghapus fase A–C.

In [ ]:
# === EDIT ME ===========================================================
REPO_URL   = ''
REPO_DIR   = 'foundation-cp'
DRIVE_ROOT = '/content/drive/MyDrive/pcc'

# SUMBER DUMP. LTC sudah punya ID gdown konkret (dari notebook 00) -> nol langkah
# manual. CCC belum: ID-nya harus dibaca dari download_data.sh mereka. Jadi survei
# LTC dulu; pindah ke CCC hanya kalau tidak ada dump LTC yang memenuhi premis.
# TIDAK ADA saklar sumber. Versi sebelumnya punya SOURCE yang harus diedit manual,
# dan default-nya ('ltc') sudah diketahui GAGAL premis — jadi setiap run default
# berakhir dengan assert. Sekarang SEMUA sumber disurvei dalam satu jalan dan yang
# terbaik dipakai. Tabel perbandingannya sendiri adalah temuan yang dicari.
CCC_DATASETS = ('imagenet',)   # tambah 'inaturalist' (iNat-2021, 633 kelas) bila perlu
GID_SCORES_LTC = {'plantnet': '1k_PPQV3VJT44hz02CcnbqPstjQo70vGr',
                  'inaturalist': '1W8R8Jj2bhS2PbR-3X9vEw-WkanbOk6mq'}
# ID CCC dari download_data.sh mereka. Ditanam supaya TIDAK ada langkah manual:
# versi sebelumnya hanya mencetak skripnya dan menyuruh menjalankan gdown sendiri,
# yang membingungkan dan tidak perlu begitu ID-nya diketahui.
GID_SCORES_CCC = {'imagenet':   '1AQjUn3m010N_i6-sfD690W7mZq2RTwJz',   # 4,62 GB
                  'inaturalist':'1BUlQZhS_5x2LJpyxCGI1IkmRrkvmRD88',   # 6,72 GB (iNat-2021, 633 kelas)
                  'places365':  '119k7PE1l72fg5Rpez5brIOn28BwClqv2',   # 0,54 GB (365 kelas)
                  'cifar-100':  '1yXD9XqBxEJnJxcfnnduNK6nHHUU3iX_6'}   # 0,01 GB (100 kelas)
# Catatan daya uji: premis butuh >=500 kelas layak, jadi places365 (365 kelas) dan
# cifar-100 (100 kelas) TIDAK BISA memenuhinya secara konstruksi, berapa pun
# sampel per kelasnya. Yang mungkin: imagenet (1.000) dan iNat-2021 (633).
LTC_DATASETS = ('inaturalist', 'plantnet')   # rilis memuat varian -trunc juga
LOSS_VARIANT = 'cross_entropy'  # 'cross_entropy' | 'focal'. LTC mengirim SEMBILAN
                                # berkas dengan NAMA IDENTIK di subdirektori berbeda;
                                # tercampur = skor dari model lain, akurasi mirip,
                                # kalibrasi beda total. Notebook 00 kena isu yang sama.
# places365 (365 kelas) dan cifar-100 (100 kelas) TIDAK BISA memenuhi premis >=500
# kelas secara konstruksi, berapa pun sampel per kelasnya — jadi tidak diunduh.
N_CLASSES_EXPECTED = None      # None = jangan dipaksakan; dibaca dari dump

# --- PRIMER, ditetapkan di prereg_imagenet_gate.md. JANGAN diubah setelah melihat hasil.
ALPHA_PRIMARY = 0.10
N_CAL_PRIMARY = 25
N_BOOT_CLASS  = 400           # bootstrap tingkat-kelas untuk gate B
N_PERM_CLASS  = 1000          # permutasi tingkat-kelas untuk gate C (p_min = 1/1001)
STABLE_THRESHOLD = 0.90

# --- SEKUNDER
ALPHAS_SECONDARY = (0.01, 0.05)
N_CAL_SECONDARY  = 50
N_SPLITS_BC = 100
N_SPLITS_A  = 100
RUN_CLUSTERED_CP = True       # reproduksi baseline pada skor yang sama

FRAC_DESC, FRAC_CAL = 0.40, 0.30   # sisanya EVAL

# ANGGARAN BARIS. Dump ImageNet CCC nyata adalah (1.153.051 x 1.000) float32 = 4,61 GB
# -- sepuluh kali lebih besar dari yang tercatat di release_audit.md. Memuatnya penuh
# lalu membuat salinan turunan (thr_lac, entropi, np.partition) melewati RAM Colab.
#
# Subsampling di sini BUKAN perubahan kriteria: premis butuh >=84 sampel/kelas dan
# anggaran ini menyisakan ~230/kelas. Ia diambil sebagai FRAKSI per kelas, bukan cap
# tetap, karena cap tetap membuat semua hitungan kelas SAMA -> log_prevalence konstan
# -> ablasi prevalensi jadi hampa. Fraksi mempertahankan struktur prevalensinya.
MAX_ROWS = 250_000            # None = pakai seluruh dump
SEED = 42
# =======================================================================
print(f'PRIMER: alpha={ALPHA_PRIMARY} n_cal={N_CAL_PRIMARY} '
      f'n_boot={N_BOOT_CLASS} n_perm={N_PERM_CLASS}')


## 2. Mount + repo + env


In [ ]:
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')
if REPO_URL and not os.path.isdir(REPO_DIR):
    subprocess.run(['git','clone',REPO_URL,REPO_DIR], check=True)
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR if os.path.isabs(REPO_DIR) else '/content/'+REPO_DIR)
os.environ['PYTHONPATH'] = os.getcwd() + os.pathsep + os.environ.get('PYTHONPATH','')
subprocess.run(['pip','install','-q','-r','requirements.txt'], check=False)
subprocess.run(['pip','install','-q','gdown'], check=False)
from pcc.utils.seed import set_seed; from pcc.utils.io import environment_stamp
set_seed(SEED)
print('env:', environment_stamp()['packages'])


## 3. Siapkan SEMUA dump — TANPA citra, TANPA GPU

Dump LTC dipakai lokasi yang sama dengan notebook 00 (`released_scores/<dataset>`), jadi kalau
sudah ada tidak diunduh ulang. Dump CCC diunduh otomatis dengan ID yang sudah ditanam.

Keduanya disurvei berdampingan di sel 4. Itu bukan pemborosan: **perbandingan ekor-panjang
versus berimbang adalah temuannya**, dan menurunkannya dari satu tabel lebih kuat daripada dari
dua run terpisah.


In [ ]:
import glob, zipfile, numpy as np

def ltc_dir(ds):
    return f'{DRIVE_ROOT}/released_scores/{ds}'

def ccc_dir(ds):
    return f'{DRIVE_ROOT}/scores_ccc/{ds}'

for ds in LTC_DATASETS:
    d = ltc_dir(ds); os.makedirs(d, exist_ok=True)
    if glob.glob(f'{d}/**/*_softmax.npy', recursive=True):
        print(f'ltc/{ds}: sudah ada, dilewati')
        continue
    print(f'ltc/{ds}: mengunduh...')
    z = f'{d}/{ds}.zip'
    r = subprocess.run(['gdown', GID_SCORES_LTC[ds], '-O', z], capture_output=True, text=True)
    if r.returncode:
        print('  gdown gagal:', r.stderr.strip()[:300])
    else:
        subprocess.run(['unzip','-o','-q',z,'-d',d], check=False)

def inventory(d, label):
    files = [p for p in sorted(glob.glob(f'{d}/**/*', recursive=True)) if os.path.isfile(p)]
    print(f'  isi {label}: {len(files)} berkas')
    for p in files[:25]:
        print(f'    {os.path.relpath(p, d):56s} {os.path.getsize(p)/1e6:9.2f} MB')
    return files

def looks_like_html(p):
    # Kegagalan kuota Google Drive menulis halaman HTML DAN mengembalikan kode 0.
    # Inilah sebabnya returncode tidak boleh dipercaya sebagai bukti unduhan berhasil.
    try:
        with open(p, 'rb') as fh:
            head = fh.read(400)
    except OSError:
        return False, b''
    low = head.lower()
    return (b'<html' in low or b'<!doctype html' in low), head

for ds in CCC_DATASETS:
    d = ccc_dir(ds); os.makedirs(d, exist_ok=True)
    mat = f'/content/ccc_npy/{ds}'
    # Tiga keadaan, dan versi sebelumnya hanya mengenali yang pertama:
    #   (a) .npy sudah dimaterialkan di /content -> tidak ada kerja
    #   (b) .npz ada di Drive tapi .npy hilang (sesi baru; /content ephemeral)
    #       -> ekstrak ulang, JANGAN unduh 4,6 GB lagi
    #   (c) tidak ada apa pun -> unduh
    # Pemeriksaan lama hanya mencari .npy DI DRIVE, yang tidak pernah ada karena
    # materialisasinya ke /content. Jadi setiap sesi baru mengunduh ulang 4,6 GB.
    if glob.glob(f'{mat}/*.npy') or glob.glob(f'{d}/**/*.npy', recursive=True):
        print(f'ccc/{ds}: .npy sudah ada, dilewati')
        continue
    if glob.glob(f'{d}/*.npz'):
        print(f'ccc/{ds}: .npz ada di Drive, ekstrak ulang tanpa mengunduh')
    else:
        print(f'ccc/{ds}: mengunduh (beberapa GB, sabar)...')
    if not glob.glob(f'{d}/*.npz'):
        r = subprocess.run(['gdown', '--fuzzy', GID_SCORES_CCC[ds]],
                           cwd=d, capture_output=True, text=True)
        if r.stdout.strip():
            print('  stdout:', r.stdout.strip()[-500:])
        if r.stderr.strip():
            print('  stderr:', r.stderr.strip()[-300:])
        print(f'  returncode: {r.returncode}  <- BUKAN bukti; diverifikasi di bawah')

    for p in sorted(glob.glob(f'{d}/*')):
        low = p.lower()
        if low.endswith('.zip'):
            subprocess.run(['unzip','-o','-q',p,'-d',d], check=False)
        elif low.endswith(('.tar.gz','.tgz','.tar')):
            subprocess.run(['tar','-xf',p,'-C',d], check=False)

    # .npz ADALAH zip berisi beberapa .npy. Versi sebelumnya mencocokkan ekstensi
    # secara literal ('.zip'/'.tar'), jadi imagenet.npz dilewati dan tidak ada .npy
    # terbentuk -- padahal unduhannya berhasil penuh. Satu baris, kegagalan bisu.
    #
    # Header .npy dibaca lewat zipfile TANPA mendekompresi isinya, supaya bentuk dan
    # dtype tiap anggota terlihat tanpa memuat gigabyte. Lalu HANYA dua array yang
    # dibutuhkan dimaterialkan, dan ke /content (ephemeral) bukan Drive -- mengekstrak
    # seluruh 4,6 GB ke Drive akan menggandakan pemakaian kuota tanpa alasan.
    for p in sorted(glob.glob(f'{d}/*.npz')):
        print(f'  membaca header {os.path.basename(p)} (tanpa dekompresi)...')
        members = []
        with zipfile.ZipFile(p) as zf:
            for nm in zf.namelist():
                try:
                    with zf.open(nm) as fh:
                        ver = np.lib.format.read_magic(fh)
                        if ver == (1, 0):
                            shp, _fo, dt = np.lib.format.read_array_header_1_0(fh)
                        elif ver == (2, 0):
                            shp, _fo, dt = np.lib.format.read_array_header_2_0(fh)
                        else:
                            continue
                    members.append((nm, shp, dt))
                    print(f'    {nm:36s} {str(shp):20s} {dt}')
                except Exception as e:
                    print(f'    {nm:36s} header tak terbaca: {type(e).__name__}')

        twod = [m for m in members if len(m[1]) == 2]
        oned = [m for m in members if len(m[1]) == 1]
        if not twod or not oned:
            print('  npz ini tidak memuat pasangan (2-D, 1-D) — kirim daftar di atas.')
            continue
        sc = max(twod, key=lambda m: m[1][0] * m[1][1])
        lb = next((m for m in oned if m[1][0] == sc[1][0]), None)
        if lb is None:
            print(f'  tidak ada array 1-D sepanjang {sc[1][0]} untuk mendampingi {sc[0]}')
            continue
        out = f'/content/ccc_npy/{ds}'
        os.makedirs(out, exist_ok=True)
        print(f'  materialkan {sc[0]} {sc[1]} dan {lb[0]} {lb[1]} -> {out}')
        # zipfile.namelist() memberi nama DENGAN sufiks '.npy', tetapi NpzFile
        # diindeks TANPA sufiks -> z['softmax.npy'] KeyError, z['softmax'] benar.
        # Ditemukan oleh tes sintetik sebelum run nyata.
        k_sc = sc[0][:-4] if sc[0].endswith('.npy') else sc[0]
        k_lb = lb[0][:-4] if lb[0].endswith('.npy') else lb[0]
        with np.load(p, allow_pickle=False) as z:
            np.save(f'{out}/scores.npy', z[k_sc])
            np.save(f'{out}/labels.npy', z[k_lb])

    files = inventory(d, f'ccc/{ds}')
    npys = (glob.glob(f'{d}/**/*.npy', recursive=True)
            + glob.glob(f'/content/ccc_npy/{ds}/*.npy'))
    if not npys:
        print(f'  GAGAL: tidak ada .npy terbentuk untuk ccc/{ds}.')
        for p in files:
            is_html, head = looks_like_html(p)
            if is_html:
                print(f'  PENYEBAB: {os.path.basename(p)} adalah HALAMAN HTML, bukan data.')
                print('  Itu batas kuota Google Drive; gdown tetap keluar dengan kode 0.')
                print('  cuplikan:', ' '.join(head.decode('utf-8','replace').split())[:300])
                print(f'  URL: https://drive.google.com/uc?id={GID_SCORES_CCC[ds]}')
                break
        else:
            print('  Berkas ADA tetapi tidak menghasilkan .npy — kirim daftar di atas.')
    else:
        print(f'  OK: {len(npys)} .npy siap dipakai')

# dua akar untuk CCC: Drive (kalau .npy langsung) dan /content (hasil materialisasi
# anggota .npz). Keduanya diperiksa supaya tidak peduli bentuk rilisnya.
roots = ([(ltc_dir(ds), 'ltc', ds) for ds in LTC_DATASETS]
         + [(ccc_dir(ds), 'ccc', ds) for ds in CCC_DATASETS]
         + [(f'/content/ccc_npy/{ds}', 'ccc', ds) for ds in CCC_DATASETS])
found = []
for rt, src, ds in roots:
    for f in sorted(glob.glob(f'{rt}/**/*.npy', recursive=True)):
        found.append((f, src, ds))
print()
print(f'total .npy: {len(found)}')
for f, src, ds in found[:60]:
    a = np.load(f, mmap_mode='r')
    print(f'  [{src}] {os.path.basename(f):52s} {str(a.shape):18s} {a.dtype}')
if not found:
    print('TIDAK ADA .npy sama sekali. Isi direktori mentah:')
    for rt, src, ds in roots:
        for p in sorted(glob.glob(f'{rt}/**/*', recursive=True))[:25]:
            if os.path.isfile(p):
                print(f'  {os.path.relpath(p, DRIVE_ROOT):64s} {os.path.getsize(p)/1e6:8.1f} MB')


### 3b. Referensi — dari mana ID CCC berasal (opsional, tidak perlu dijalankan)

ID di sel 1 diambil dari `download_data.sh` milik CCC. Sel ini hanya untuk memverifikasi
bahwa ID-nya belum berubah; ia **tidak diperlukan** untuk menjalankan notebook.


In [ ]:
SHOW_CCC_SCRIPT = False
if SHOW_CCC_SCRIPT:
    if not os.path.isdir('/content/ccc'):
        subprocess.run(['git','clone','--depth','1',
                        'https://github.com/tiffanyding/class-conditional-conformal.git',
                        '/content/ccc'], check=False)
    print(open('/content/ccc/download_data.sh').read())
    print('bandingkan dengan GID_SCORES_CCC di sel 1')
else:
    print('dilewati (ID sudah ditanam di sel 1)')


## 5. Dump dan kepala

In [ ]:
import glob

def _pair(scores):
    for suf in ('_softmax.npy', '_scores.npy', 'scores.npy'):
        if scores.endswith(suf):
            cand = scores[: -len(suf)] + suf.replace('softmax', 'labels').replace(
                'scores', 'labels')
            if os.path.exists(cand):
                return cand
    cand = os.path.join(os.path.dirname(scores), 'labels.npy')
    return cand if os.path.exists(cand) else None

DUMPS = {}

# CCC: materialisasi sel di atas menaruhnya di /content/ccc_npy/<ds>/
for p in sorted(glob.glob('/content/ccc_npy/*/scores.npy')):
    ds = os.path.basename(os.path.dirname(p))
    lab = _pair(p)
    if lab:
        DUMPS['ccc_' + ds] = {'scores': p, 'labels': lab, 'eval_scores': None,
                              'eval_labels': None, 'max_rows': MAX_ROWS}

# LTC: pasangkan cal (DESC+CAL) dengan test (EVAL penuh)
for ds in LTC_DATASETS:
    d = f'{DRIVE_ROOT}/released_scores/{ds}'
    cal = sorted(glob.glob(f'{d}/**/*cal_softmax.npy', recursive=True))
    tst = sorted(glob.glob(f'{d}/**/*test_softmax.npy', recursive=True))
    cal = [p for p in cal if LOSS_VARIANT in p or LOSS_VARIANT == 'cross_entropy']
    tst = [p for p in tst if LOSS_VARIANT in p or LOSS_VARIANT == 'cross_entropy']
    if cal and tst and _pair(cal[0]) and _pair(tst[0]):
        DUMPS['ltc_' + ds] = {'scores': cal[0], 'labels': _pair(cal[0]),
                              'eval_scores': tst[0], 'eval_labels': _pair(tst[0]),
                              'max_rows': None}

assert DUMPS, 'tidak ada dump siap pakai -- periksa sel penyiapan di atas'
for k, v in DUMPS.items():
    a = np.load(v['scores'], mmap_mode='r')
    line = '  ' + k.ljust(18) + ' cal ' + str(a.shape)
    if v['eval_scores']:
        e = np.load(v['eval_scores'], mmap_mode='r')
        line += '  eval ' + str(e.shape) + '  (dump terpisah)'
    else:
        line += '  eval = 30% dari dump yang sama'
    print(line)

In [ ]:
import numpy as np, os, glob, subprocess

HEAD_DIR = '/content/head'
os.makedirs(HEAD_DIR, exist_ok=True)
HEADS = {}          # K -> (path W, path b)

def _register(tag, W, b):
    w_p = f'{HEAD_DIR}/{tag}_fc_weight.npy'
    b_p = f'{HEAD_DIR}/{tag}_fc_bias.npy'
    np.save(w_p, W)
    np.save(b_p, np.zeros(len(W)) if b is None else b)
    HEADS[int(W.shape[0])] = (w_p, b_p)
    print('  kepala', tag, W.shape, '-> K =', W.shape[0])

# --- 1. torchvision ResNet-50 (ImageNet-1k). Model tersupervisi yang BERBEDA dari
# SimCLRv2+probe penghasil skor CCC: ketidakcocokan itu justru yang membuat phi
# eksogen sepenuhnya terhadap delta_y.
tv_w = f'{HEAD_DIR}/torchvision_imagenet_fc_weight.npy'
if os.path.exists(tv_w):
    W = np.load(tv_w)
    HEADS[int(W.shape[0])] = (tv_w, tv_w.replace('_weight', '_bias'))
    print('  kepala torchvision_imagenet sudah ada -> K =', W.shape[0])
else:
    from pcc.descriptors.head_weights import load_torchvision_resnet50_head
    W, b = load_torchvision_resnet50_head()
    _register('torchvision_imagenet', W, b)

# --- 2. Kepala LTC untuk Pl@ntNet dan iNat-2018. Run pertama Phase 2 gagal di kedua
# dataset itu, tetapi HANYA keluarga ruang-output yang pernah dijalankan di sana --
# dan keluarga itu juga gagal di ImageNet. Jadi itu kegagalan KELUARGA phi, bukan
# kegagalan dataset, dan checkpoint yang dirilis membuatnya bisa diuji tanpa GPU.
GID_MODELS = '1tS-M-4IYyCGMeIxxyrgx2-XCZgdvw18S'   # models.zip, dari notebook 00
CKPT_DIR = f'{DRIVE_ROOT}/checkpoints/ltc_models'
os.makedirs(CKPT_DIR, exist_ok=True)
if not glob.glob(f'{CKPT_DIR}/**/*model*.pth', recursive=True):
    print('mengunduh models.zip LTC (6 ResNet-50)...')
    subprocess.run(['gdown', GID_MODELS, '-O', f'{CKPT_DIR}/models.zip'], check=True)
    subprocess.run(['unzip', '-o', f'{CKPT_DIR}/models.zip', '-d', CKPT_DIR],
                   check=True)

def _variant_ok(path):
    # PERANGKAP dari notebook 00: LTC mengirim ENAM model dengan NAMA BERKAS
    # IDENTIK dan menaruh varian focal di subdirektori 'focal_loss'. Glob rekursif
    # bisa mengambil mana saja, jadi checkpoint dan skor bisa diam-diam berasal dari
    # varian berbeda -- akurasi mirip, kalibrasi beda total.
    is_focal = 'focal' in path.replace(chr(92), '/').lower()
    return is_focal if LOSS_VARIANT == 'focal' else (not is_focal)

from pcc.data.ltc_datasets import NUM_CLASSES
for ds in LTC_DATASETS:
    tag = 'ltc_' + ds
    w_p = f'{HEAD_DIR}/{tag}_fc_weight.npy'
    if os.path.exists(w_p):
        W = np.load(w_p, mmap_mode='r')
        HEADS[int(W.shape[0])] = (w_p, w_p.replace('_weight', '_bias'))
        print('  kepala', tag, 'sudah ada -> K =', W.shape[0])
        continue
    cands = sorted(glob.glob(f'{CKPT_DIR}/**/best-{ds}-model.pth', recursive=True))
    keep = [p for p in cands if _variant_ok(p)]
    print('  checkpoint', ds, ':', len(cands), 'kandidat,', len(keep),
          'cocok varian', LOSS_VARIANT)
    for p in cands:
        print('     ' + ('* ' if _variant_ok(p) else '  ') + p)
    if not keep:
        print('     DILEWATI: tidak ada checkpoint varian', LOSS_VARIANT)
        continue
    try:
        from pcc.extract.backbones import load_ltc_resnet50
        m = load_ltc_resnet50(keep[0], NUM_CLASSES[ds], None)
        W = m.fc.weight.detach().cpu().numpy()
        b = m.fc.bias.detach().cpu().numpy() if m.fc.bias is not None else None
        del m
        assert W.shape[0] == NUM_CLASSES[ds], (W.shape, NUM_CLASSES[ds])
        _register(tag, W, b)
    except Exception as e:
        print('     GAGAL memuat:', type(e).__name__, str(e)[:160])

print()
print('kepala tersedia per jumlah kelas:', {k: os.path.basename(v[0])
                                            for k, v in sorted(HEADS.items())})

## 6. Penyimpan Drive — dipanggil setelah tiap fase

Ukuran diverifikasi per berkas; berkas yang sudah utuh dilewati, jadi panggilan
berikutnya nyaris gratis.

In [ ]:
import glob, shutil, time
RUN_DIR = DRIVE_ROOT + '/runs/nb09_' + time.strftime('%Y%m%d_%H%M%S')
os.makedirs(RUN_DIR, exist_ok=True)
print('tujuan:', RUN_DIR)

def save_to_drive(tag):
    n_ok = n_bad = 0
    for p in sorted(glob.glob('pcc/reports/*.json')):
        d = os.path.join(RUN_DIR, os.path.basename(p))
        try:
            if os.path.exists(d) and os.path.getsize(d) == os.path.getsize(p):
                n_ok += 1
                continue
            shutil.copy2(p, d)
            ok = os.path.getsize(d) == os.path.getsize(p)
            n_ok += int(ok)
            n_bad += int(not ok)
        except Exception as e:
            n_bad += 1
            print('   gagal', os.path.basename(p), e)
    print('   [{}] tersimpan {} gagal {} -> Drive'.format(tag, n_ok, n_bad))
    return n_bad == 0

save_to_drive('awal')

## 7. Runner bersama

Satu fungsi untuk semua fase. Tiap konfigurasi menulis laporannya sendiri lewat
`write_report`, jadi jejaknya identik dengan menjalankan skrip dari CLI, dan satu
kegagalan hanya menghapus satu sel tabel.

In [ ]:
import traceback
from pcc.experiments import phase2_pcc as drv
from pcc.utils.io import write_report

PRIMARY = 'ccc_imagenet'      # 1168 baris/kelas -- satu-satunya yang bisa disapu dalam
ALPHA   = 0.10                # yang bisa disapu dalam, dan alpha tempat PCC lolos
SEEDS   = (0, 1, 2, 3, 4)
RESULTS, FAILED = [], []

class A: pass

def build(dump, phi, seed, **over):
    v = DUMPS[dump]
    x = A()
    x.scores, x.labels = v['scores'], v['labels']
    x.eval_scores, x.eval_labels = v['eval_scores'], v['eval_labels']
    x.max_rows = v['max_rows']
    x.dataset = dump
    x.reports_dir = 'pcc/reports'
    x.alpha, x.n_cal = ALPHA, 25
    x.heldout_frac = 0.30
    x.frac_desc = 0.0 if phi == 'head' else 0.40
    x.frac_cal = 0.30
    x.phi = phi
    K = int(np.load(v['scores'], mmap_mode='r').shape[1])
    hw, hb = HEADS.get(K, (None, None))
    x.head_weights = hw if phi == 'head' else None
    x.head_bias = hb if phi == 'head' else None
    x.distance_holdout = 'w_cos_knn_1' if phi == 'head' else 'prof_knn_1'
    x.stat = 'worst'
    x.ccc_root = '/content/ccc' if os.path.isdir('/content/ccc') else None
    x.seed = seed
    x.name = None
    x.print_json = False
    x.cal_depth = None
    x.lam_override = None
    x.n_star_rule = 'oos'
    x.no_recalibrate = False
    x.feature_group = 'all'
    for k, val in over.items():
        assert hasattr(x, k), 'atribut tak dikenal: ' + k   # salah ketik = diam
        setattr(x, k, val)
    return x

def run_one(phase, tag, dump, phi, seed, **over):
    t0 = time.time()
    try:
        x = build(dump, phi, seed, **over)
        r = drv.run(x)
        c = drv.verdict(r, x.stat)
        nm = 'nb09_{}_{}_s{}'.format(phase, tag, seed).replace('.', 'p')
        write_report('pcc/reports', nm, hypothesis=drv.HYPOTHESIS,
                     pass_criteria=drv.PASS_CRITERIA, config=vars(x),
                     seed=seed, results=r, conclusion=c, started_at=t0)
        RESULTS.append(dict(phase=phase, tag=tag, dump=dump, phi=phi, seed=seed,
                            secs=time.time() - t0, res=r, conclusion=c))
        t2 = r.get('table_2_heldout')
        s2 = t2['primary_stat'] if t2 else None
        print('  {:26s} s{} {:5.0f}s | lam {:.3f} | T2 {:+.4f} | {}'.format(
            tag, seed, time.time() - t0, r['pcc']['lambda'],
            t2['delta'].get(s2, float('nan')) if t2 else float('nan'), c),
            flush=True)
        return r
    except Exception as e:
        FAILED.append(dict(phase=phase, tag=tag, seed=seed,
                           error=type(e).__name__ + ': ' + str(e)))
        print('  {:26s} s{} GAGAL: {}'.format(tag, seed, str(e)[:150]), flush=True)
        traceback.print_exc()
        return None

print('runner siap | dump primer', PRIMARY, '| alpha', ALPHA)

## FASE A — sapuan kedalaman kalibrasi

**Eksperimen paling bernilai dalam daftar.** Empat setting berbeda pendapat apakah
PCC bekerja, dan satu-satunya pembedanya baris CAL per kelas (76 / 20 / 12 / 3) —
tetapi itu terancu dengan dataset **dan** backbone sekaligus.

Membatasi kedalaman **di dalam satu dump** membuang kedua ancu. CCC ImageNet punya
1168 baris/kelas, jadi 10 → 200 menyapu langsung melewati 76.

Kalau kurvanya monoton dan menyeberang nol di sekitar 40–76, satu kemenangan kita
berubah dari anomali menjadi **titik pada garis yang bisa diprediksi.** Kalau datar,
penjelasan kedalaman-kalibrasi salah dan itu yang dilaporkan.

In [ ]:
DEPTHS = (10, 25, 50, 100, 200)
print('FASE A: kedalaman', DEPTHS, '| phi kepala | seed', SEEDS)
for d in DEPTHS:
    for s in SEEDS:
        run_one('A_depth', 'depth{}'.format(d), PRIMARY, 'head', s,
                cal_depth=d, n_cal=min(25, max(5, d // 2)))
save_to_drive('fase A')

## FASE B — sapuan fraksi held-out

Berapa banyak label space boleh tanpa data kalibrasi sebelum ekstrapolasinya runtuh.

In [ ]:
FRACS = (0.10, 0.30, 0.50)
print('FASE B: fraksi held-out', FRACS)
for f in FRACS:
    for s in SEEDS:
        run_one('B_heldout', 'ho{}'.format(f), PRIMARY, 'head', s, heldout_frac=f)
save_to_drive('fase B')

## FASE C — ablasi

Menjawab satu pertanyaan yang belum punya jawaban sama sekali: **bagian mana yang
sebenarnya bekerja?** `lam0` harus memberi Tabel 2 tepat nol (koreksinya hilang);
`lam1` adalah δ̂ mentah yang Amandemen 8 ukur **merugikan**; `prevalence` adalah
lengan prediktor-trivial yang harus dikalahkan.

In [ ]:
ABL = [('full',        {}),
       ('lam0',        dict(lam_override=0.0)),
       ('lam1',        dict(lam_override=1.0)),
       ('no_recal',    dict(no_recalibrate=True)),
       ('nstar_mse',   dict(n_star_rule='mse')),
       ('feat_dist',   dict(feature_group='distance')),
       ('feat_prev',   dict(feature_group='prevalence')),
       ('feat_noprev', dict(feature_group='no_prevalence'))]
print('FASE C: ablasi', [a for a, _ in ABL])
for tag, over in ABL:
    for s in SEEDS:
        run_one('C_abl', tag, PRIMARY, 'head', s, **over)
save_to_drive('fase C')

## FASE D — pesaing terdekat, API DITEMUKAN bukan ditebak

§7 menyebut **Fuzzy Classwise CP** dan **PAS/Interp-Q** paling berbahaya: keduanya
memakai kemiripan kelas yang ditentukan tangan, jadi pesaing konseptual terdekat PCC.

API repo LTC **belum pernah didaftar**. Menebak signature sudah dua kali salah di
proyek ini, jadi sel ini meng-clone, **mencari** modulnya, lalu mencetak signature
tiap fungsi. Kalau tidak cocok dengan harapan, itu terlihat **di sini**, bukan merusak
grid di atasnya.

In [ ]:
import sys
LTC_URL = 'https://github.com/tiffanyding/long-tail-conformal.git'
LTC_DIR = '/content/ltc'
ltc_api = {}
try:
    import importlib, inspect
    if not (os.path.isdir(LTC_DIR) and os.listdir(LTC_DIR)):
        subprocess.run(['git', 'clone', '--depth', '1', LTC_URL, LTC_DIR], check=True)
    print('isi', LTC_DIR, '->', sorted(os.listdir(LTC_DIR))[:20])
    hits = sorted(glob.glob(LTC_DIR + '/**/*conformal*.py', recursive=True))
    print('modul kandidat:', [os.path.relpath(h, LTC_DIR) for h in hits])
    if hits:
        root = os.path.dirname(os.path.dirname(hits[0]))
        pkg = os.path.basename(os.path.dirname(hits[0]))
        if root not in sys.path:
            sys.path.insert(0, root)
        importlib.invalidate_caches()
        print('akar impor:', root, '| paket:', pkg)
        for h in hits:
            mod = pkg + '.' + os.path.basename(h)[:-3]
            try:
                m = importlib.import_module(mod)
            except Exception as e:
                ltc_api[mod] = 'gagal impor: {}: {}'.format(type(e).__name__, e)
                continue
            fns = []
            for nm, ob in vars(m).items():
                if nm.startswith('_') or not callable(ob):
                    continue
                if getattr(ob, '__module__', None) != mod:
                    continue
                try:
                    fns.append(nm + str(inspect.signature(ob)))
                except (ValueError, TypeError):
                    fns.append(nm + '(?)')
            ltc_api[mod] = sorted(fns)
    for mod, api in ltc_api.items():
        print(' ', mod)
        for f in (api if isinstance(api, list) else [api]):
            print('    ', f[:150])
except Exception as e:
    ltc_api = {'error': '{}: {}'.format(type(e).__name__, e)}
    print('gagal:', ltc_api['error'])
print()
print('Wiring pemanggilannya ditulis dari daftar DI ATAS, bukan dari ingatan.')
save_to_drive('fase D (API)')

## FASE E — uji berpasangan, Holm, effect size

Mesinnya sudah ada sejak nb05 tapi tidak pernah dipakai di Phase 2. Reviewer uSxM
secara spesifik meminta effect size pada paper lama.

In [ ]:
from pcc.eval.stats import mean_ci, holm_bonferroni
from collections import defaultdict

def arm(phase, tag):
    out = []
    for r in RESULTS:
        if r['phase'] != phase or r['tag'] != tag:
            continue
        t2 = r['res'].get('table_2_heldout')
        if t2 and t2['primary_stat'] in t2['delta']:
            out.append(t2['delta'][t2['primary_stat']])
    return np.array(out, float)

base = arm('C_abl', 'full')
print('FASE E: pembanding = ablasi full, n =', len(base))
STATS, pv, lb = {}, [], []
for tag, _ in ABL:
    if tag == 'full':
        continue
    other = arm('C_abl', tag)
    n = min(len(base), len(other))
    if n < 3:
        continue
    d = base[:n] - other[:n]
    ci = mean_ci(d)
    sd = float(np.std(d, ddof=1))
    dz = float(ci['mean'] / sd) if sd > 0 else float('nan')   # Cohen's d_z
    rng = np.random.default_rng(0)
    boot = np.array([np.mean(rng.choice(d, n, replace=True)) for _ in range(10000)])
    # min(1, .) is not cosmetic: when every paired difference is exactly zero both
    # tails are 1.0 and the doubled two-sided p comes out 2.0 -- a number that
    # cannot be a p-value and would go straight into a table. The dry-run printed
    # p=2.0000 for two ablations, which is also the signal that those two arms are
    # BIT-IDENTICAL to full, so that is recorded as a fact rather than as a test.
    p = float(min(1.0, 2 * min((boot <= 0).mean(), (boot >= 0).mean())))
    ident = bool(np.all(d == 0))
    STATS[tag] = {'mean_diff': ci['mean'], 'ci': [ci['ci_low'], ci['ci_high']],
                  'cohen_dz': dz, 'p_boot': p, 'n': n, 'identical_to_full': ident}
    pv.append(p); lb.append(tag)
    print('  full - {:12s} {:+.4f} [{:+.4f},{:+.4f}] d_z={:+.2f} p={:.4f}{}'.format(
        tag, ci['mean'], ci['ci_low'], ci['ci_high'], dz, p,
        '  <- IDENTIK dengan full' if ident else ''))
if pv:
    holm = holm_bonferroni(pv, alpha=0.05)
    for t, h in zip(lb, holm):
        STATS[t]['holm'] = h
        print('  Holm {:12s} p={:.4f} ambang={:.4f} tolak={}'.format(
            t, h['p_value'], h['threshold'], h['reject']))

# Rekalibrasi marginal adalah SATU skalar yang ditambahkan ke semua ambang, dan
# pencocokan ukuran-set membatalkan konstanta itu dengan tepat. Jadi ablasi E3
# mustahil terbaca di tabel matched -- bukan karena tidak berefek, tetapi karena
# tampilan itu buta terhadapnya. Efeknya dibaca di ambang MENTAH, tempat validitas
# marginal sebenarnya tinggal.
print()
print('=== E3 pada ambang MENTAH (tak dicocokkan ukurannya) ===')
for tag in ('full', 'no_recal'):
    got = [r for r in RESULTS if r['phase'] == 'C_abl' and r['tag'] == tag
           and r['res'].get('table_2_heldout')]
    if not got:
        continue
    vals = [r['res']['table_2_heldout']['raw_unmatched']['pcc'] for r in got]
    mc = float(np.mean([v['marginal_cov'] for v in vals]))
    sz = float(np.mean([v['avg_set_size'] for v in vals]))
    # offset dicetak karena tanpa itu dua baris yang kebetulan sama terbaca sebagai
    # bug. Kalau offset ~ 0, kedua lengan MEMANG ambang yang sama, dan itu fakta
    # tentang datanya (lambda=0 -> ambang sudah kuantil global), bukan tentang kode.
    off = float(np.mean([r['res']['pcc']['offset'] for r in got]))
    print('  {:9s} offset {:+.5f} | cakupan marginal {:.4f} (target {:.2f}) |'
          ' ukuran set {:.3f}'.format(tag, off, mc, 1 - ALPHA, sz))
save_to_drive('fase E')

## FASE F — runtime, dan ringkasan akhir

PCC **tidak punya training**: g_θ ridge atas ≤1000 titik dengan ≤15 fitur. Itu klaim
praktis yang membedakan dari Clustered CP (butuh clustering) dan RC3P.

In [ ]:
secs = defaultdict(list)
for r in RESULTS:
    secs[r['phase']].append(r['secs'])
RUNTIME = {k: {'mean_s': float(np.mean(v)), 'n': len(v)} for k, v in secs.items()}
for k, v in sorted(RUNTIME.items()):
    print('  {:12s} {:6.1f}s rata-rata x{}'.format(k, v['mean_s'], v['n']))

SUMMARY = defaultdict(dict)
for r in RESULTS:
    key = r['phase'] + '|' + r['tag']
    for tn in ('table_1_seen', 'table_2_heldout'):
        tb = r['res'].get(tn)
        if not tb:
            continue
        st = tb['primary_stat']
        for lab, val in (('delta_' + st, tb['delta'].get(st)),
                         ('oracle_' + st, tb.get('delta_oracle', {}).get(st)),
                         ('marginal_cov', tb['pcc'].get('marginal_cov')),
                         ('avg_set_size', tb['pcc'].get('avg_set_size')),
                         ('sscv', tb['pcc'].get('sscv'))):
            if val is not None and np.isfinite(val):
                SUMMARY[key].setdefault(tn + '|' + lab, []).append(float(val))
SUMMARY = {k: {m: mean_ci(np.array(v, float)) for m, v in d.items()}
           for k, d in SUMMARY.items()}

print()
print('=== FASE A: kurva kedalaman kalibrasi (Tabel 2) ===')
for d in DEPTHS:
    k = 'A_depth|depth{}'.format(d)
    c = SUMMARY.get(k, {}).get('table_2_heldout|delta_worst')
    o = SUMMARY.get(k, {}).get('table_2_heldout|oracle_worst')
    if c:
        # 'x% dari plafon' hanya punya arti kalau plafonnya positif. Dry-run
        # menghasilkan oracle -0.0885 dan rumus lama mencetak '437%' -- angka
        # yang terbaca sebagai kemenangan besar padahal ruangnya memang tidak ada.
        om = o['mean'] if o else float('nan')
        pct = ('{:.0f}% dari plafon'.format(100 * c['mean'] / om)
               if o and om > 1e-6 else 'plafon <= 0: rasio tak bermakna')
        print('  n_cal<={:4d}  {:+.4f} [{:+.4f},{:+.4f}]  oracle {:+.4f}  {}'
              .format(d, c['mean'], c['ci_low'], c['ci_high'], om, pct))

CAVEATS = [
    'Sapuan kedalaman & held-out di CCC ImageNet saja -- satu-satunya dump yang',
    '  cukup dalam (1168 baris/kelas) untuk disapu tanpa ancu dataset/backbone.',
    'alpha=0.10: alpha tempat PCC lolos; r_delta turun 0.829->0.738->0.620 seiring',
    '  alpha mengecil, jadi batas alpha punya mekanisme (lihat nb06).',
    'Pesaing LTC: API DIDAFTAR di fase D, pemanggilannya BELUM -- wiring ditulis',
    '  dari daftar itu, dan Sec 7 tetap belum terpenuhi sampai direproduksi.',
    'Oracle memakai label EVAL: plafon, bukan metode.',
]
for c in CAVEATS:
    print('CAVEAT:', c)

path = write_report('pcc/reports', '09_wave1_summary',
                    hypothesis=drv.HYPOTHESIS, pass_criteria=drv.PASS_CRITERIA,
                    config={'primary': PRIMARY, 'alpha': ALPHA,
                            'seeds': list(SEEDS), 'depths': list(DEPTHS),
                            'heldout_fracs': list(FRACS),
                            'ablations': [a for a, _ in ABL], 'seed': SEED},
                    seed=SEED,
                    results={'summary': SUMMARY, 'paired_tests': STATS,
                             'runtime': RUNTIME, 'ltc_api': ltc_api,
                             'n_ok': len(RESULTS), 'n_failed': len(FAILED),
                             'failed': FAILED, 'caveats': CAVEATS},
                    conclusion='SELESAI' if not FAILED else 'SEBAGIAN',
                    started_at=time.time())
print('laporan:', path)
save_to_drive('final')
z = shutil.make_archive(RUN_DIR, 'zip', RUN_DIR)
print('zip', z, '{:.1f} MB'.format(os.path.getsize(z) / 1e6))